# Taller: El Problema de la Mochila Multidimensional

**Curso:** Ciencia de Datos - Resolución de Problemas del Mundo Real

Pontificia Universidad Católica del Ecuador

**Nombre:**

---

## Objetivos de aprendizaje

Al finalizar este taller el estudiante será capaz de:

1. Reconocer el problema de la mochila clásica y entender su versión **multidimensional (MKP)**.
2. Formular el MKP como un modelo de **programación lineal entera (PLE)** con variables binarias.
3. Implementar y resolver el modelo en Python usando **Gurobi**.
4. Interpretar la solución óptima: qué se selecciona, qué recursos quedan limitantes y cuáles tienen holgura.
5. Aplicar el modelo a un caso real de **selección de proyectos** con varios recursos.

## Agenda (2 horas)

| Tiempo | Actividad |
|--------|-----------|
| 0:00 a 0:15 | Repaso de la mochila clásica y motivación |
| 0:15 a 0:35 | El problema multidimensional: definición y formulación |
| 0:35 a 1:00 | Ejemplo guiado resuelto en Gurobi (lo corremos juntos) |
| 1:00 a 1:10 | Pausa y preguntas |
| 1:10 a 1:50 | Ejercicio práctico (lo implementan ustedes) |
| 1:50 a 2:00 | Discusión de resultados y cierre |


# Parte 1. Teoría y ejemplo guiado

## 1.1 Repaso: la mochila clásica (0/1 Knapsack)

Imagina que tienes una mochila con una capacidad máxima de peso $C$ y un conjunto de $n$ objetos. Cada objeto $j$ tiene un **valor** $p_j$ y un **peso** $w_j$. Quieres elegir qué objetos llevar para **maximizar el valor total** sin exceder la capacidad.

Definimos una variable binaria por objeto:

$$
x_j = \begin{cases} 1 & \text{si se lleva el objeto } j \\ 0 & \text{en caso contrario} \end{cases}
$$

El modelo clásico es:

\begin{align}
\max \quad & \sum_{j=1}^{n} p_j\, x_j \\
\text{s.a.} \quad & \sum_{j=1}^{n} w_j\, x_j \leq C \\
& x_j \in \{0,1\}, \quad j = 1,\dots,n
\end{align}

La idea central: una sola restricción de capacidad (el peso).

## 1.2 El problema de la mochila multidimensional (MKP)

En la práctica, rara vez existe un único recurso limitante. Una decisión de selección suele estar restringida por **varios recursos a la vez**: peso *y* volumen, presupuesto *y* horas de trabajo *y* capacidad de cómputo, etc.

El **Problema de la Mochila Multidimensional** (Multidimensional Knapsack Problem, MKP) generaliza la mochila clásica: en lugar de una sola restricción de capacidad, tenemos $m$ restricciones, una por cada **dimensión** o tipo de recurso.

**Aplicaciones típicas:**

* Selección de proyectos o portafolios de inversión (presupuesto, personal, riesgo).
* Carga de contenedores o vehículos (peso, volumen, número de bultos).
* Asignación de recursos de cómputo (memoria, CPU, GPU).
* Selección de features en machine learning bajo varios presupuestos de costo.

## 1.3 Formulación matemática del MKP

**Conjuntos**

* $J = \{1, 2, \dots, n\}$: conjunto de objetos candidatos.
* $I = \{1, 2, \dots, m\}$: conjunto de recursos (las dimensiones).

**Parámetros**

* $p_j$: valor (o beneficio) del objeto $j$, para todo $j \in J$.
* $w_{ij}$: cantidad del recurso $i$ que consume el objeto $j$, para todo $i \in I$, $j \in J$.
* $c_i$: capacidad disponible del recurso $i$, para todo $i \in I$.

**Variables de decisión**

$$
x_j = \begin{cases} 1 & \text{si se selecciona el objeto } j \\ 0 & \text{en caso contrario} \end{cases}
\qquad \forall j \in J
$$

**Modelo**

\begin{align}
\max \quad & \sum_{j \in J} p_j\, x_j \\
\text{s.a.} \quad & \sum_{j \in J} w_{ij}\, x_j \leq c_i, \quad \forall i \in I \\
& x_j \in \{0,1\}, \quad \forall j \in J
\end{align}

La única diferencia con la mochila clásica es que ahora hay $m$ restricciones de capacidad en lugar de una.

## 1.4 Ejemplo guiado

Un excursionista prepara una mochila con capacidad de **50 kg** de peso y **40 litros** de volumen. Tiene 6 objetos candidatos:

| Objeto | Valor | Peso (kg) | Volumen (L) |
|--------|-------|-----------|-------------|
| 1 | 60  | 10 | 8  |
| 2 | 100 | 20 | 12 |
| 3 | 120 | 30 | 25 |
| 4 | 80  | 15 | 10 |
| 5 | 40  | 5  | 4  |
| 6 | 70  | 12 | 9  |

In [ ]:
! pip install gurobipy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 53.7 MB/s eta 0:00:00


In [ ]:
from gurobipy import Model, GRB, quicksum
import pandas as pd
import numpy as np

In [ ]:
# Datos del ejemplo guiado
valor   = [60, 100, 120, 80, 40, 70]
peso    = [10,  20,  30, 15,  5, 12]
volumen = [ 8,  12,  25, 10,  4,  9]

W         = np.array([peso, volumen])   # (m, n) = (2, 6)
capacidad = [50, 40]
m, n      = W.shape

print('Recursos (dimensiones):', m)
print('Objetos candidatos:', n)

Recursos (dimensiones): 2
Objetos candidatos: 6


In [ ]:
modelo = Model('mochila_multidimensional')

x = {j: modelo.addVar(vtype=GRB.BINARY, name='x' + str(j)) for j in range(n)}

modelo.setObjective(quicksum(valor[j] * x[j] for j in range(n)), GRB.MAXIMIZE)

for i in range(m):
    modelo.addConstr(
        quicksum(W[i, j] * x[j] for j in range(n)) <= capacidad[i],
        name='recurso' + str(i)
    )

modelo.update()
modelo.optimize()

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 2 rows, 6 columns and 12 nonzeros (Max)
Model fingerprint: 0xa2d6eeba
Model has 6 linear objective coefficients
Variable types: 0 continuous, 6 integer (6 binary)
Coefficient statistics:
  Matrix range     [4e+00, 3e+01]
  Objective range  [4e+01, 1e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [4e+01, 5e+01]

Found heuristic solution: objective 280.0000000
Presolve time: 0.00s
Presolved: 2 rows, 6 columns, 12 nonzeros
Variable types: 0 continuous, 6 integer (6 binary)

Root relaxation: cutoff, 0 iterations, 0.00 seconds (0.00 work units)

Explored 1 nodes (0 simplex iterations) in 0.02 seconds (0.00 work units)
Thread count was 2 (of 2 available processors)

Solution count 1: 280 

Optimal solution found (tolerance 1.00e

In [ ]:
print('Valor total óptimo:', int(modelo.ObjVal))
print()
print('Objetos seleccionados:')
for j in range(n):
    if x[j].X > 0.5:
        print(f'  Objeto {j+1}: valor {valor[j]}, peso {peso[j]}, volumen {volumen[j]}')

print()
nombres_recursos = ['Peso (kg)', 'Volumen (L)']
for i in range(m):
    usado = sum(W[i, j] * x[j].X for j in range(n))
    print(f'  {nombres_recursos[i]}: {usado:.0f} usado de {capacidad[i]} disponible')

Valor total óptimo: 280

Objetos seleccionados:
  Objeto 1: valor 60, peso 10, volumen 8
  Objeto 2: valor 100, peso 20, volumen 12
  Objeto 4: valor 80, peso 15, volumen 10
  Objeto 5: valor 40, peso 5, volumen 4

  Peso (kg): 50 usado de 50 disponible
  Volumen (L): 34 usado de 40 disponible


### Interpretación

El valor óptimo es **280**, seleccionando los objetos **1, 2, 4 y 5**.

* El **peso** queda en 50 de 50: restricción **activa** (cuello de botella).
* El **volumen** queda en 34 de 40: hay holgura de 6 litros.

El objeto 3 tiene el mayor valor individual (120) pero consume 30 kg de los 50 disponibles, dejando solo 20 kg para el resto, lo que resulta en un valor total menor.

---
# Parte 2. Ejercicio práctico

## Contexto: selección de proyectos de I+D

Una empresa de tecnología debe decidir **qué proyectos de investigación y desarrollo financiar**. Hay **10 proyectos candidatos** con retorno esperado (miles USD) y consumo en tres recursos:

| Recurso | Capacidad disponible |
|---------|----------------------|
| Presupuesto | 300 miles USD |
| Horas de ingeniería | 2200 |
| Unidades de cómputo (GPU) | 70 |

In [ ]:
# Lectura de datos desde GitHub
datos       = pd.read_csv('https://raw.githubusercontent.com/danirodriguez4657/CD-an-lisis-de-problemas-reales/refs/heads/main/knapsack_proyectos.csv')
caps        = pd.read_csv('https://raw.githubusercontent.com/danirodriguez4657/CD-an-lisis-de-problemas-reales/refs/heads/main/knapsack_capacidades.csv')
capacidades = dict(zip(caps['recurso'], caps['capacidad']))

print('Capacidades disponibles:', capacidades)
datos

Capacidades disponibles: {'presupuesto': 300, 'horas': 2200, 'gpu': 70}


,proyecto,retorno,presupuesto,horas,gpu
0,P1,120,40,300,8
1,P2,200,75,520,14
2,P3,150,55,410,10
3,P4,90,30,220,5
4,P5,310,95,640,20
5,P6,175,60,480,12
6,P7,260,85,600,18
7,P8,130,45,350,9
8,P9,145,50,390,11
9,P10,280,90,620,19


## Paso 1. Conjuntos y parámetros

In [ ]:
# Paso 1: conjuntos y parámetros
N       = len(datos)
retorno = datos['retorno'].values
recursos = ['presupuesto', 'horas', 'gpu']

# Matriz de consumo W: filas = recursos, columnas = proyectos
W = np.array([
    datos['presupuesto'].values,
    datos['horas'].values,
    datos['gpu'].values
])  # forma (3, 10)

cap = [capacidades[r] for r in recursos]  # [300, 2200, 70]
m   = len(cap)

print(f'Proyectos (n): {N}')
print(f'Recursos  (m): {m}')
print(f'Capacidades : {dict(zip(recursos, cap))}')

Proyectos (n): 10
Recursos  (m): 3
Capacidades : {'presupuesto': 300, 'horas': 2200, 'gpu': 70}


## Paso 2. Variables de decisión

In [ ]:
# Paso 2: modelo y variables de decisión
# Variable binaria: aprobar (1) o no aprobar (0) cada proyecto
mkp = Model('MKP_proyectos')

y = {j: mkp.addVar(vtype=GRB.BINARY, name=f'y_{datos.iloc[j]["proyecto"]}')
     for j in range(N)}

print(f'Variables creadas: {N} variables binarias')

Variables creadas: 10 variables binarias


## Paso 3. Función objetivo

In [ ]:
# Paso 3: función objetivo — maximizar retorno total
mkp.setObjective(
    quicksum(retorno[j] * y[j] for j in range(N)),
    GRB.MAXIMIZE
)

## Paso 4. Restricciones

In [ ]:
# Paso 4: restricciones de capacidad — una por cada recurso (3 en total)
for i in range(m):
    mkp.addConstr(
        quicksum(W[i, j] * y[j] for j in range(N)) <= cap[i],
        name=f'capacidad_{recursos[i]}'
    )

mkp.update()
print(f'Restricciones agregadas: {mkp.NumConstrs}')

Restricciones agregadas: 3


## Paso 5. Resolver

In [ ]:
# Paso 5: resolver el modelo
mkp.optimize()

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 3 rows, 10 columns and 30 nonzeros (Max)
Model fingerprint: 0x7cbdaca3
Model has 10 linear objective coefficients
Variable types: 0 continuous, 10 integer (10 binary)
Coefficient statistics:
  Matrix range     [5e+00, 6e+02]
  Objective range  [9e+01, 3e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [7e+01, 2e+03]

Found heuristic solution: objective 870.0000000
Presolve removed 1 rows and 0 columns
Presolve time: 0.00s
Presolved: 2 rows, 10 columns, 20 nonzeros
Variable types: 0 continuous, 10 integer (10 binary)

Root relaxation: objective 9.400000e+02, 1 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    Be

## Paso 6. Presentar la solución

In [ ]:
# Paso 6: presentar la solución
print('=' * 50)
print(f'Retorno total óptimo: {int(mkp.ObjVal)} miles USD')
print('=' * 50)

print('\nProyectos seleccionados:')
print('-' * 50)
seleccionados = []
for j in range(N):
    if y[j].X > 0.5:
        proy = datos.iloc[j]
        seleccionados.append(proy['proyecto'])
        print(f"  {proy['proyecto']}: retorno={proy['retorno']:>4}, "
              f"presupuesto={proy['presupuesto']:>3}, "
              f"horas={proy['horas']:>4}, gpu={proy['gpu']:>3}")

print(f'\nTotal proyectos aprobados: {len(seleccionados)}')

print('\nUso de recursos:')
print('-' * 50)
for i in range(m):
    usado   = sum(W[i, j] * y[j].X for j in range(N))
    holgura = cap[i] - usado
    estado  = '⚠️  LIMITANTE' if holgura < 1 else f'holgura = {holgura:.0f}'
    print(f'  {recursos[i]:12s}: {usado:>5.0f} / {cap[i]}  →  {estado}')

Retorno total óptimo: 940 miles USD

Proyectos seleccionados:
--------------------------------------------------
  P4: retorno=  90, presupuesto= 30, horas= 220, gpu=  5
  P5: retorno= 310, presupuesto= 95, horas= 640, gpu= 20
  P7: retorno= 260, presupuesto= 85, horas= 600, gpu= 18
  P10: retorno= 280, presupuesto= 90, horas= 620, gpu= 19

Total proyectos aprobados: 4

Uso de recursos:
--------------------------------------------------
  presupuesto :   300 / 300  →  ⚠️  LIMITANTE
  horas       :  2080 / 2200  →  holgura = 120
  gpu         :    62 / 70  →  holgura = 8


## Paso 7. Conclusiones

* Se seleccionaron **4 proyectos** (P4, P5, P7, P10) con un **retorno total óptimo de 940 miles USD**.
* El recurso **presupuesto** resultó ser el **cuello de botella**: se agotaron los 300 miles USD disponibles por completo, sin holgura.
* Los recursos **horas** (holgura = 120) y **GPU** (holgura = 8) no fueron limitantes.
* El proyecto **P5** tiene el mayor retorno individual (310) y fue seleccionado. Sin embargo, proyectos como P2 (retorno 200) quedaron fuera porque su combinación de recursos junto con los demás seleccionados excedería el presupuesto disponible.
* Si se dispusiera de más presupuesto, sería posible incorporar proyectos adicionales de alto retorno como P2 o P6.

---
## Desafío adicional

### Desafío 1: Proyectos incompatibles (P5 y P7)
P5 y P7 usan el mismo equipo especializado y no pueden ejecutarse simultáneamente.

**Restricción a agregar:** $y_{P5} + y_{P7} \leq 1$

In [ ]:
# Desafío 1: P5 y P7 incompatibles
mkp1 = Model('MKP_d1')
y1 = {j: mkp1.addVar(vtype=GRB.BINARY) for j in range(N)}
mkp1.setObjective(quicksum(retorno[j]*y1[j] for j in range(N)), GRB.MAXIMIZE)
for i in range(m):
    mkp1.addConstr(quicksum(W[i,j]*y1[j] for j in range(N)) <= cap[i])

# Restricción de incompatibilidad
idx5 = list(datos['proyecto']).index('P5')
idx7 = list(datos['proyecto']).index('P7')
mkp1.addConstr(y1[idx5] + y1[idx7] <= 1, name='incompatible_P5_P7')

mkp1.update()
mkp1.optimize()

sel1 = [datos.iloc[j]['proyecto'] for j in range(N) if y1[j].X > 0.5]
print(f'Retorno óptimo: {int(mkp1.ObjVal)} miles USD')
print(f'Proyectos: {sel1}')
print(f'Cambio vs base: {int(mkp1.ObjVal) - 940} miles USD')

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 4 rows, 10 columns and 32 nonzeros (Max)
Model fingerprint: 0x4e464ead
Model has 10 linear objective coefficients
Variable types: 0 continuous, 10 integer (10 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+02]
  Objective range  [9e+01, 3e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+03]

Found heuristic solution: objective 870.0000000
Presolve removed 1 rows and 0 columns
Presolve time: 0.00s
Presolved: 3 rows, 10 columns, 22 nonzeros
Variable types: 0 continuous, 10 integer (10 binary)

Root relaxation: objective 9.312500e+02, 3 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    Be

### Desafío 2: Dependencia (P3 solo si P1)

In [ ]:
# Desafío 2: P3 solo puede aprobarse si también se aprueba P1
mkp2 = Model('MKP_d2')
y2 = {j: mkp2.addVar(vtype=GRB.BINARY) for j in range(N)}
mkp2.setObjective(quicksum(retorno[j]*y2[j] for j in range(N)), GRB.MAXIMIZE)
for i in range(m):
    mkp2.addConstr(quicksum(W[i,j]*y2[j] for j in range(N)) <= cap[i])

# Restricción de dependencia
idx1 = list(datos['proyecto']).index('P1')
idx3 = list(datos['proyecto']).index('P3')
mkp2.addConstr(y2[idx3] <= y2[idx1], name='dependencia_P3_P1')

mkp2.update()
mkp2.optimize()

sel2 = [datos.iloc[j]['proyecto'] for j in range(N) if y2[j].X > 0.5]
print(f'Retorno óptimo: {int(mkp2.ObjVal)} miles USD')
print(f'Proyectos: {sel2}')
print(f'Cambio vs base: {int(mkp2.ObjVal) - 940} miles USD')

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 4 rows, 10 columns and 32 nonzeros (Max)
Model fingerprint: 0x21b4f063
Model has 10 linear objective coefficients
Variable types: 0 continuous, 10 integer (10 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+02]
  Objective range  [9e+01, 3e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [7e+01, 2e+03]

Found heuristic solution: objective 870.0000000
Presolve removed 1 rows and 0 columns
Presolve time: 0.00s
Presolved: 3 rows, 10 columns, 22 nonzeros
Variable types: 0 continuous, 10 integer (10 binary)

Root relaxation: objective 9.400000e+02, 1 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    Be

### Desafío 3: Cardinalidad máxima (a lo sumo 3 proyectos)

In [ ]:
# Desafío 3: máximo 3 proyectos aprobados
mkp3 = Model('MKP_d3')
y3 = {j: mkp3.addVar(vtype=GRB.BINARY) for j in range(N)}
mkp3.setObjective(quicksum(retorno[j]*y3[j] for j in range(N)), GRB.MAXIMIZE)
for i in range(m):
    mkp3.addConstr(quicksum(W[i,j]*y3[j] for j in range(N)) <= cap[i])

# Restricción de cardinalidad
mkp3.addConstr(quicksum(y3[j] for j in range(N)) <= 3, name='max_3_proyectos')

mkp3.update()
mkp3.optimize()

sel3 = [datos.iloc[j]['proyecto'] for j in range(N) if y3[j].X > 0.5]
print(f'Retorno óptimo: {int(mkp3.ObjVal)} miles USD')
print(f'Proyectos: {sel3}')
print(f'Cambio vs base: {int(mkp3.ObjVal) - 940} miles USD')

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 4 rows, 10 columns and 40 nonzeros (Max)
Model fingerprint: 0x15dbf97e
Model has 10 linear objective coefficients
Variable types: 0 continuous, 10 integer (10 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+02]
  Objective range  [9e+01, 3e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [3e+00, 2e+03]

Found heuristic solution: objective 470.0000000
Presolve removed 4 rows and 10 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of 2 available processors)

Solution count 2: 850 470 

Optimal solution found (tolerance 1.00e-04)
Best objective 8.500000000000e+02, best bound 8.500000000000e+02, gap 0.0

### Resumen de los desafíos

| Variante | Retorno (miles USD) | Proyectos seleccionados | Cambio vs base |
|----------|--------------------|--------------------------|-----------------|
| Base | 940 | P4, P5, P7, P10 | — |
| D1: P5 y P7 incompatibles | 930 | P1, P4, P5, P8, P10 | −10 |
| D2: P3 depende de P1 | 940 | P4, P5, P7, P10 | 0 |
| D3: máx 3 proyectos | 850 | P5, P7, P10 | −90 |

**Observaciones:**
- El **Desafío 2** no afecta el resultado porque P3 no era parte de la solución óptima base.
- El **Desafío 1** reduce ligeramente el retorno (−10) forzando sustituir P7 por otros proyectos.
- El **Desafío 3** tiene el mayor impacto (−90) al limitar drásticamente la cantidad de proyectos.

## Preguntas de reflexión final

1. **¿Por qué el MKP usa variables binarias y no continuas?** Porque cada proyecto se aprueba o no completamente — no existe financiamiento parcial. Una solución fraccionaria como $y_{P5} = 0.7$ no tiene sentido real: no se puede ejecutar el 70% de un proyecto.

2. **¿Qué implica la complejidad NP-difícil cuando el número de proyectos crece?** Con cientos o miles de proyectos, el espacio de búsqueda crece exponencialmente y los solvers exactos como Gurobi pueden tardar horas o no encontrar el óptimo en tiempo razonable. En esos casos se recurre a heurísticas o metaheurísticas.

3. **¿Cómo conecta con feature selection en machine learning?** En feature selection bajo múltiples presupuestos, cada feature es un objeto, su aporte al modelo es el valor, y los costos computacionales (tiempo, memoria, latencia) son las dimensiones del MKP.